In [1]:
import os
import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader, random_split

In [2]:
data_base_dir = "/data/CIFAR-10-C"

In [3]:
data_base_dir = "/data/CIFAR-10-C"

if not os.path.exists(data_base_dir):
    # Try relative to notebook location (playground/) first, then from project root
    candidate = "./data/CIFAR-10-C"
    if os.path.exists(candidate):
        data_base_dir = candidate
    else:
        data_base_dir = "./playground/data/CIFAR-10-C"

print("Using data dir:", data_base_dir)

Using data dir: ./data/CIFAR-10-C


In [4]:
x_path = os.path.join(data_base_dir, "gaussian_noise.npy")
y_path = os.path.join(data_base_dir, "labels.npy")

X = np.load(x_path)     # expected shape: (50000, 32, 32, 3)
y = np.load(y_path)     # expected shape: (50000,)

print("X shape:", X.shape, "dtype:", X.dtype)
print("y shape:", y.shape, "dtype:", y.dtype)
print("num classes:", len(np.unique(y)))

X shape: (50000, 32, 32, 3) dtype: uint8
y shape: (50000,) dtype: uint8
num classes: 10


In [5]:
X_tensor = torch.tensor(X, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0  # (N, 3, 32, 32)
y_tensor = torch.tensor(y, dtype=torch.long)

In [6]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),                             # 16x16
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),                             # 8x8
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 256), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

In [8]:
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct = 0, 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y_batch)
        correct += (logits.argmax(1) == y_batch).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0, 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)
            total_loss += criterion(logits, y_batch).item() * len(y_batch)
            correct += (logits.argmax(1) == y_batch).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

In [9]:
# ── CONFORMAL PREDICTION ──────────────────────────────────────────────────────
import math

ALPHA = 0.1

# 1. split: train / calibration / test
dataset = TensorDataset(X_tensor, y_tensor)
n = len(dataset)
train_size = int(0.6 * n)
cal_size   = int(0.2 * n)
test_size  = n - train_size - cal_size

train_ds, cal_ds, test_ds = random_split(dataset, [train_size, cal_size, test_size])

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
cal_loader   = DataLoader(cal_ds,   batch_size=64)
test_loader  = DataLoader(test_ds,  batch_size=64)

In [10]:
# 2. (Re-)train model
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(10):
    train_loss, train_acc = train(model, train_loader, optimizer, criterion)
    val_loss, val_acc     = evaluate(model, cal_loader, criterion)
    print(f"Epoch {epoch+1:2d} | Train {train_loss:.4f}/{train_acc:.3f} | Cal {val_loss:.4f}/{val_acc:.3f}")

Epoch  1 | Train 1.6854/0.387 | Cal 1.3717/0.501
Epoch  2 | Train 1.2630/0.542 | Cal 1.0898/0.609
Epoch  3 | Train 1.0292/0.635 | Cal 0.8844/0.700
Epoch  4 | Train 0.8467/0.701 | Cal 0.7372/0.750
Epoch  5 | Train 0.6905/0.758 | Cal 0.6000/0.799
Epoch  6 | Train 0.5577/0.804 | Cal 0.4598/0.849
Epoch  7 | Train 0.4503/0.843 | Cal 0.3825/0.881
Epoch  8 | Train 0.3748/0.868 | Cal 0.3585/0.884
Epoch  9 | Train 0.3160/0.891 | Cal 0.2756/0.921
Epoch 10 | Train 0.2658/0.907 | Cal 0.2306/0.933


In [11]:
# 3. Calibration – collect nonconformity scores s_i = 1 - ŷ[true_class]
model.eval()
cal_scores = []
with torch.no_grad():
    for X_batch, y_batch in cal_loader:
        probs = torch.softmax(model(X_batch.to(device)), dim=1).cpu()
        true_class_probs = probs[torch.arange(len(y_batch)), y_batch]
        cal_scores.append(1.0 - true_class_probs)

cal_scores = torch.cat(cal_scores)          # shape: (n_cal,)

# Finite-sample corrected quantile  (Angelopoulos & Bates 2022, eq. 2)
n_cal = len(cal_scores)
q_level = math.ceil((n_cal + 1) * (1 - ALPHA)) / n_cal
q_hat = torch.quantile(cal_scores, min(q_level, 1.0))
print(f"\nCalibration quantile q̂ = {q_hat:.4f}  (α={ALPHA}, n_cal={n_cal})")


Calibration quantile q̂ = 0.4289  (α=0.1, n_cal=10000)


In [12]:
# 4. Test – build prediction sets and measure coverage / efficiency
model.eval()
covered, total, set_sizes = 0, 0, []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        probs = torch.softmax(model(X_batch.to(device)), dim=1).cpu()
        # prediction set: all classes with score ≤ q̂  ↔  prob ≥ 1 - q̂
        pred_sets = probs >= (1.0 - q_hat)

        for i, y_true in enumerate(y_batch):
            in_set = pred_sets[i, y_true.item()].item()
            covered += int(in_set)
            set_sizes.append(pred_sets[i].sum().item())
        total += len(y_batch)

empirical_coverage = covered / total
avg_set_size       = sum(set_sizes) / len(set_sizes)
print(f"Empirical coverage : {empirical_coverage:.4f}  (target ≥ {1-ALPHA:.2f})")
print(f"Avg prediction-set size: {avg_set_size:.3f}  (out of 10 classes)")

Empirical coverage : 0.8956  (target ≥ 0.90)
Avg prediction-set size: 0.935  (out of 10 classes)


In [13]:
# 5. Per-class coverage
classes = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']
print("\nPer-class coverage:")
class_covered = [0]*10;  class_total = [0]*10
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        probs = torch.softmax(model(X_batch.to(device)), dim=1).cpu()
        pred_sets = probs >= (1.0 - q_hat)
        for i, y_true in enumerate(y_batch):
            c = y_true.item()
            class_covered[c] += int(pred_sets[i, c].item())
            class_total[c]   += 1
for i, cls in enumerate(classes):
    print(f"  {cls:<12}: {class_covered[i]/class_total[i]:.3f}")


Per-class coverage:
  airplane    : 0.912
  automobile  : 0.975
  bird        : 0.820
  cat         : 0.903
  deer        : 0.839
  dog         : 0.871
  frog        : 0.891
  horse       : 0.916
  ship        : 0.954
  truck       : 0.886
